# AlpCAN -- CXR Pipeline: MedSAM Segmentasyon Baseline

**Amac:** NIH ChestX-ray14 goruntuleri uzerinde MedSAM (Medical Segment Anything Model) ile akciger segmentasyonu ve nodul lokalizasyon performansini degerlendirmek. AlpCAN CXR Pipeline Agent 7D (MedRAX segmentasyon bileseni) baseline.

**Icerik:**
1. Veri seti yukleme -- NIH ChestX-ray14 + BBox annotasyonlari
2. MedSAM model yukleme (HuggingFace)
3. Akciger bolgesi segmentasyonu
4. BBox-guided nodul/kitle segmentasyonu
5. Segmentasyon kalite metrikleri (Dice, IoU)
6. CXR siniflandirma sonuclariyla birlesik analiz
7. Gorsellestirme ve sonuc kayit

**Dataset:** `nih-chest-xrays/data` (45GB, orijinal cozunurluk)  
**Model:** MedSAM ViT-Base (flaviagiammarino/medsam-vit-base)  
**GPU:** Kaggle T4 16GB

In [ ]:
!pip install -q transformers torch torchvision scikit-learn

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import torch
from transformers import SamModel, SamProcessor
from sklearn.metrics import jaccard_score

warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"CUDA: {torch.cuda.is_available()}")

---
## 1. Veri Seti Yukleme -- NIH ChestX-ray14 + BBox Annotasyonlari

In [ ]:
# Dataset yolunu bul
INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")

# Data_Entry CSV
csv_candidates = list(INPUT_DIR.rglob("Data_Entry*.csv"))
if not csv_candidates:
    raise FileNotFoundError("Data_Entry CSV bulunamadi! Dataset eklenmi mi?")
LABELS_CSV = csv_candidates[0]

# Goruntu klasorleri (images_001 - images_012)
all_img_dirs = sorted(INPUT_DIR.rglob("images"))
IMG_DIRS = [d for d in all_img_dirs if d.is_dir() and list(d.glob('*.png'))]

if not IMG_DIRS:
    all_pngs = list(INPUT_DIR.rglob("*.png"))
    if all_pngs:
        IMG_DIRS = list(set(p.parent for p in all_pngs[:10]))

print(f"Labels CSV: {LABELS_CSV}")
print(f"Image directories: {len(IMG_DIRS)}")
for d in IMG_DIRS[:5]:
    n_files = len(list(d.glob('*.png')))
    print(f"  {d}: {n_files:,} PNG")

In [ ]:
# Tum goruntu yollarini index'le
image_index = {}
for img_dir in IMG_DIRS:
    for png in img_dir.glob('*.png'):
        image_index[png.name] = png

print(f"Toplam indekslenen goruntu: {len(image_index):,}")

In [ ]:
# Labels CSV'yi oku
df = pd.read_csv(LABELS_CSV)
print(f"CSV kayit sayisi: {len(df):,}")
print(f"Hasta sayisi: {df['Patient ID'].nunique():,}")

# 14 patoloji etiketlerini binary sutunlara donustur
ALL_LABELS = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Effusion', 'Emphysema', 'Fibrosis', 'Hernia',
    'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening',
    'Pneumonia', 'Pneumothorax'
]

for label in ALL_LABELS:
    df[label] = df['Finding Labels'].apply(lambda x: 1 if label in str(x) else 0)

df['No Finding'] = df['Finding Labels'].apply(lambda x: 1 if x == 'No Finding' else 0)

print(f"\nPatoloji Dagilimi:")
label_counts = df[ALL_LABELS].sum().sort_values(ascending=False)
for label, count in label_counts.items():
    pct = count / len(df) * 100
    marker = " << AlpCAN" if label in ('Nodule', 'Mass') else ""
    print(f"  {label:>20s}: {count:>6,} ({pct:>5.1f}%){marker}")

In [ ]:
# BBox verisi -- lokalize edilmis bulgular
bbox_candidates = list(INPUT_DIR.rglob("BBox_List*.csv"))
if bbox_candidates:
    bbox_df = pd.read_csv(bbox_candidates[0])
    print(f"BBox kayit sayisi: {len(bbox_df):,}")
    print(f"BBox sutunlari: {list(bbox_df.columns)}")
    print(f"\nBBox patoloji dagilimi:")
    print(bbox_df['Finding Label'].value_counts())
    
    # BBox sutun isimlerini normalize et
    bbox_cols = list(bbox_df.columns)
    # Beklenen: Image Index, Finding Label, x, y, w, h (son 4 sutun koordinatlar)
    coord_cols = bbox_cols[-4:]  # Son 4 sutun bbox koordinatlari
    print(f"\nBBox koordinat sutunlari: {coord_cols}")
    bbox_df.head()
else:
    print("BBox verisi bulunamadi")
    bbox_df = None

---
## 2. MedSAM Model Yukleme

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SamModel.from_pretrained("flaviagiammarino/medsam-vit-base").to(device)
processor = SamProcessor.from_pretrained("flaviagiammarino/medsam-vit-base")

print(f"MedSAM loaded on {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Model type: {type(model).__name__}")

---
## 3. Akciger Bolgesi Segmentasyonu

In [ ]:
def segment_lung_region(image_path, model, processor, device):
    """Segment lung region using MedSAM with full-image bounding box."""
    img = Image.open(image_path).convert('RGB')
    w, h = img.size
    # Full chest bounding box (covering lung area)
    input_boxes = [[int(w * 0.1), int(h * 0.05), int(w * 0.9), int(h * 0.85)]]
    inputs = processor(img, input_boxes=[input_boxes], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs, multimask_output=False)
    masks = processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(), inputs["original_sizes"].cpu(),
        inputs["reshaped_input_sizes"].cpu()
    )
    return masks[0][0][0].numpy()  # Binary mask

In [ ]:
# Hizli test -- tek goruntu
test_name = list(image_index.keys())[0]
test_path = image_index[test_name]
print(f"Test goruntusu: {test_name}")

start = time.time()
test_mask = segment_lung_region(test_path, model, processor, device)
elapsed = time.time() - start

print(f"Mask shape: {test_mask.shape}")
print(f"Mask range: [{test_mask.min()}, {test_mask.max()}]")
print(f"Segmented area: {test_mask.sum():,} pixels ({test_mask.sum() / test_mask.size * 100:.1f}%)")
print(f"Inference time: {elapsed:.2f}s")

# Gorsellestir
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
img = Image.open(test_path).convert('L')
axes[0].imshow(img, cmap='gray')
axes[0].set_title('Orijinal CXR')
axes[0].axis('off')

axes[1].imshow(test_mask, cmap='jet', alpha=0.8)
axes[1].set_title('MedSAM Lung Mask')
axes[1].axis('off')

axes[2].imshow(img, cmap='gray')
axes[2].imshow(test_mask, cmap='jet', alpha=0.35)
axes[2].set_title('Overlay')
axes[2].axis('off')

plt.suptitle(f'MedSAM Akciger Segmentasyonu -- {test_name}', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. BBox-Guided Nodul/Kitle Segmentasyonu

In [ ]:
def segment_nodule(image_path, bbox, model, processor, device):
    """Segment nodule using MedSAM with annotated bounding box.
    
    Args:
        image_path: Path to CXR image
        bbox: [x, y, w, h] bounding box coordinates
        model: MedSAM model
        processor: MedSAM processor
        device: torch device
    
    Returns:
        mask: Binary segmentation mask
        score: IoU confidence score from model
    """
    img = Image.open(image_path).convert('RGB')
    # bbox = [x, y, w, h] -> [x1, y1, x2, y2]
    input_boxes = [[bbox[0], bbox[1], bbox[0] + bbox[2], bbox[1] + bbox[3]]]
    inputs = processor(img, input_boxes=[input_boxes], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs, multimask_output=True)
    masks = processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(), inputs["original_sizes"].cpu(),
        inputs["reshaped_input_sizes"].cpu()
    )
    scores = outputs.iou_scores.cpu().numpy()[0][0]
    best_idx = np.argmax(scores)
    return masks[0][0][best_idx].numpy(), scores[best_idx]


def compute_dice(pred_mask, gt_mask):
    """Compute Dice coefficient between predicted and ground truth masks."""
    pred_flat = pred_mask.astype(bool).flatten()
    gt_flat = gt_mask.astype(bool).flatten()
    intersection = np.logical_and(pred_flat, gt_flat).sum()
    if pred_flat.sum() + gt_flat.sum() == 0:
        return 1.0  # Both empty
    return 2.0 * intersection / (pred_flat.sum() + gt_flat.sum())


def compute_iou(pred_mask, gt_mask):
    """Compute IoU (Jaccard) between predicted and ground truth masks."""
    pred_flat = pred_mask.astype(bool).flatten()
    gt_flat = gt_mask.astype(bool).flatten()
    intersection = np.logical_and(pred_flat, gt_flat).sum()
    union = np.logical_or(pred_flat, gt_flat).sum()
    if union == 0:
        return 1.0  # Both empty
    return intersection / union


def bbox_to_mask(bbox, img_shape):
    """Convert [x, y, w, h] bounding box to binary mask."""
    mask = np.zeros(img_shape[:2], dtype=bool)
    x, y, w, h = int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3])
    y_end = min(y + h, img_shape[0])
    x_end = min(x + w, img_shape[1])
    mask[y:y_end, x:x_end] = True
    return mask


print("Segmentasyon ve metrik fonksiyonlari tanimlandi.")

---
## 5. BBox Annotasyonlu Goruntuleri Isle

In [ ]:
# BBox annotasyonlu Nodule/Mass goruntuleri uzerinde MedSAM calistir
if bbox_df is not None:
    # Nodule ve Mass bbox'larini filtrele
    bbox_nodule_mass = bbox_df[bbox_df['Finding Label'].isin(['Nodule', 'Mass'])].copy()
    bbox_cols = list(bbox_df.columns)
    coord_cols = bbox_cols[-4:]  # Son 4 sutun: x, y, w, h
    
    print(f"Toplam Nodule/Mass BBox annotasyonu: {len(bbox_nodule_mass):,}")
    print(f"Benzersiz goruntu: {bbox_nodule_mass['Image Index'].nunique():,}")
    
    # Mevcut goruntulerle eslestir
    bbox_nodule_mass = bbox_nodule_mass[bbox_nodule_mass['Image Index'].isin(image_index)]
    print(f"Mevcut goruntulere sahip BBox: {len(bbox_nodule_mass):,}")
    
    # Her bbox icin segmentasyon yap
    seg_results = []
    total = len(bbox_nodule_mass)
    start_time = time.time()
    
    print(f"\nSegmentasyon basliyor: {total} BBox annotasyonu")
    print("-" * 60)
    
    for i, (_, row) in enumerate(bbox_nodule_mass.iterrows()):
        img_name = row['Image Index']
        img_path = image_index.get(img_name)
        if img_path is None or not img_path.exists():
            continue
        
        try:
            # BBox koordinatlarini al
            bbox = [float(row[c]) for c in coord_cols]
            
            # MedSAM segmentasyonu
            t0 = time.time()
            pred_mask, confidence = segment_nodule(img_path, bbox, model, processor, device)
            seg_time = time.time() - t0
            
            # Ground truth mask (BBox'tan)
            gt_mask = bbox_to_mask(bbox, pred_mask.shape)
            
            # Metrikleri hesapla
            dice = compute_dice(pred_mask, gt_mask)
            iou = compute_iou(pred_mask, gt_mask)
            pred_area = pred_mask.sum()
            gt_area = gt_mask.sum()
            area_ratio = pred_area / gt_area if gt_area > 0 else 0
            
            seg_results.append({
                'image_name': img_name,
                'finding': row['Finding Label'],
                'bbox_x': bbox[0],
                'bbox_y': bbox[1],
                'bbox_w': bbox[2],
                'bbox_h': bbox[3],
                'dice': dice,
                'iou': iou,
                'confidence': confidence,
                'pred_area': int(pred_area),
                'gt_area': int(gt_area),
                'area_ratio': area_ratio,
                'seg_time': seg_time
            })
            
        except Exception as e:
            if i < 5:
                print(f"  HATA [{img_name}]: {e}")
            continue
        
        # Ilerleme raporu
        if (i + 1) % 100 == 0 or (i + 1) == total:
            elapsed = time.time() - start_time
            speed = (i + 1) / elapsed
            eta = (total - i - 1) / speed if speed > 0 else 0
            print(f"  {i+1:>5}/{total} ({(i+1)/total*100:>5.1f}%) "
                  f"| {speed:.1f} img/s | ETA: {eta:.0f}s")
    
    seg_df = pd.DataFrame(seg_results)
    total_time = time.time() - start_time
    
    print(f"\n{'=' * 60}")
    print(f"Segmentasyon tamamlandi!")
    print(f"  Islenen: {len(seg_df):,} BBox")
    print(f"  Sure: {total_time:.1f}s ({len(seg_df)/total_time:.1f} img/s)")
    print(f"  Ortalama Dice: {seg_df['dice'].mean():.4f}")
    print(f"  Ortalama IoU: {seg_df['iou'].mean():.4f}")
else:
    print("BBox verisi yok, nodul segmentasyonu atlanacak.")
    seg_df = pd.DataFrame()

---
## 6. Segmentasyon Kalite Metrikleri (Dice, IoU)

In [ ]:
if len(seg_df) > 0:
    print("=" * 65)
    print("MedSAM Nodul/Kitle Segmentasyon Metrikleri")
    print("=" * 65)
    
    # Genel metrikler
    print(f"\n{'Metrik':>25s} | {'Ortalama':>10s} | {'Medyan':>10s} | {'Std':>10s} | {'Min':>10s} | {'Max':>10s}")
    print("-" * 85)
    for metric in ['dice', 'iou', 'confidence', 'area_ratio', 'seg_time']:
        vals = seg_df[metric].values
        print(f"  {metric:>25s} | {vals.mean():>10.4f} | {np.median(vals):>10.4f} | "
              f"{vals.std():>10.4f} | {vals.min():>10.4f} | {vals.max():>10.4f}")
    
    # Patolojiye gore
    print(f"\n{'=' * 65}")
    print(f"Patolojiye Gore Sonuclar")
    print(f"{'=' * 65}")
    for finding in seg_df['finding'].unique():
        subset = seg_df[seg_df['finding'] == finding]
        print(f"\n  {finding} (n={len(subset):,}):")
        print(f"    Dice:       {subset['dice'].mean():.4f} +/- {subset['dice'].std():.4f}")
        print(f"    IoU:        {subset['iou'].mean():.4f} +/- {subset['iou'].std():.4f}")
        print(f"    Confidence: {subset['confidence'].mean():.4f} +/- {subset['confidence'].std():.4f}")
        print(f"    Area ratio: {subset['area_ratio'].mean():.4f} +/- {subset['area_ratio'].std():.4f}")
    
    # Dice dagilimi kategorizasyonu
    print(f"\n{'=' * 65}")
    print(f"Dice Skor Dagilimi")
    print(f"{'=' * 65}")
    bins = [(0.0, 0.2, 'Cok Dusuk'), (0.2, 0.4, 'Dusuk'), 
            (0.4, 0.6, 'Orta'), (0.6, 0.8, 'Iyi'), (0.8, 1.0, 'Cok Iyi')]
    for lo, hi, label in bins:
        count = ((seg_df['dice'] >= lo) & (seg_df['dice'] < hi)).sum()
        pct = count / len(seg_df) * 100
        bar = '#' * int(pct / 2)
        print(f"  [{lo:.1f}-{hi:.1f}) {label:>10s}: {count:>5} ({pct:>5.1f}%) {bar}")
else:
    print("Segmentasyon sonuclari bos.")

In [ ]:
# Metrik gorsellestirmeleri
if len(seg_df) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    
    # 1. Dice dagilimi
    axes[0, 0].hist(seg_df['dice'], bins=50, color='#3498db', edgecolor='white', alpha=0.8)
    axes[0, 0].axvline(seg_df['dice'].mean(), color='red', linestyle='--', 
                       label=f"Mean={seg_df['dice'].mean():.3f}")
    axes[0, 0].set_title('Dice Koefisyeni Dagilimi', fontweight='bold')
    axes[0, 0].set_xlabel('Dice')
    axes[0, 0].set_ylabel('Sayi')
    axes[0, 0].legend()
    
    # 2. IoU dagilimi
    axes[0, 1].hist(seg_df['iou'], bins=50, color='#2ecc71', edgecolor='white', alpha=0.8)
    axes[0, 1].axvline(seg_df['iou'].mean(), color='red', linestyle='--',
                       label=f"Mean={seg_df['iou'].mean():.3f}")
    axes[0, 1].set_title('IoU (Jaccard) Dagilimi', fontweight='bold')
    axes[0, 1].set_xlabel('IoU')
    axes[0, 1].set_ylabel('Sayi')
    axes[0, 1].legend()
    
    # 3. Confidence vs Dice scatter
    colors = ['#e74c3c' if f == 'Nodule' else '#3498db' for f in seg_df['finding']]
    axes[0, 2].scatter(seg_df['confidence'], seg_df['dice'], c=colors, alpha=0.5, s=15)
    axes[0, 2].set_xlabel('Model Confidence')
    axes[0, 2].set_ylabel('Dice')
    axes[0, 2].set_title('Confidence vs Dice', fontweight='bold')
    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', label='Nodule'),
                       Line2D([0], [0], marker='o', color='w', markerfacecolor='#3498db', label='Mass')]
    axes[0, 2].legend(handles=legend_elements)
    
    # 4. BBox boyutu vs Dice
    seg_df['bbox_area'] = seg_df['bbox_w'] * seg_df['bbox_h']
    axes[1, 0].scatter(seg_df['bbox_area'], seg_df['dice'], c=colors, alpha=0.5, s=15)
    axes[1, 0].set_xlabel('BBox Alan (piksel^2)')
    axes[1, 0].set_ylabel('Dice')
    axes[1, 0].set_title('BBox Boyutu vs Dice', fontweight='bold')
    
    # 5. Area ratio dagilimi
    axes[1, 1].hist(seg_df['area_ratio'].clip(0, 3), bins=50, color='#9b59b6', edgecolor='white', alpha=0.8)
    axes[1, 1].axvline(1.0, color='red', linestyle='--', label='1:1 oran')
    axes[1, 1].set_title('Predicted/GT Alan Orani', fontweight='bold')
    axes[1, 1].set_xlabel('Alan Orani (pred/gt)')
    axes[1, 1].set_ylabel('Sayi')
    axes[1, 1].legend()
    
    # 6. Processing time dagilimi
    axes[1, 2].hist(seg_df['seg_time'], bins=50, color='#e67e22', edgecolor='white', alpha=0.8)
    axes[1, 2].axvline(seg_df['seg_time'].mean(), color='red', linestyle='--',
                       label=f"Mean={seg_df['seg_time'].mean():.3f}s")
    axes[1, 2].set_title('Isleme Suresi Dagilimi', fontweight='bold')
    axes[1, 2].set_xlabel('Sure (s)')
    axes[1, 2].set_ylabel('Sayi')
    axes[1, 2].legend()
    
    plt.suptitle('MedSAM Segmentasyon Performans Metrikleri', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'cxr_medsam_performance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: cxr_medsam_performance.png")

---
## 7. Nodul/Kitle Segmentasyon Gorsellestirmesi

In [ ]:
# 4x3 grid: Orijinal + BBox overlay + MedSAM maske
if len(seg_df) > 0:
    # Farkli boyutlarda ornekler sec
    seg_df_sorted = seg_df.sort_values('bbox_area', ascending=False).reset_index(drop=True)
    n_examples = min(4, len(seg_df_sorted))
    
    # Farkli boyutlarda ornekler: buyuk, orta-buyuk, orta, kucuk
    indices = np.linspace(0, len(seg_df_sorted) - 1, n_examples, dtype=int)
    sample_rows = seg_df_sorted.iloc[indices]
    
    fig, axes = plt.subplots(n_examples, 3, figsize=(18, 5 * n_examples))
    if n_examples == 1:
        axes = axes[np.newaxis, :]
    
    for row_idx, (_, row) in enumerate(sample_rows.iterrows()):
        img_name = row['image_name']
        img_path = image_index.get(img_name)
        if img_path is None or not img_path.exists():
            continue
        
        bbox = [row['bbox_x'], row['bbox_y'], row['bbox_w'], row['bbox_h']]
        
        # Segmentasyonu tekrar calistir (gorsellestirme icin)
        try:
            pred_mask, confidence = segment_nodule(img_path, bbox, model, processor, device)
        except Exception:
            continue
        
        img = Image.open(img_path).convert('L')
        img_arr = np.array(img)
        
        # Sutun 1: Orijinal
        axes[row_idx, 0].imshow(img_arr, cmap='gray')
        axes[row_idx, 0].set_title(f"{img_name}\n{row['finding']}", fontsize=9)
        axes[row_idx, 0].axis('off')
        
        # Sutun 2: BBox overlay
        axes[row_idx, 1].imshow(img_arr, cmap='gray')
        rect = plt.Rectangle((bbox[0], bbox[1]), bbox[2], bbox[3],
                             linewidth=2, edgecolor='red', facecolor='none')
        axes[row_idx, 1].add_patch(rect)
        axes[row_idx, 1].set_title(f'BBox GT\n{int(bbox[2])}x{int(bbox[3])}px', fontsize=9)
        axes[row_idx, 1].axis('off')
        
        # Sutun 3: MedSAM segmentasyon
        axes[row_idx, 2].imshow(img_arr, cmap='gray')
        axes[row_idx, 2].imshow(pred_mask, cmap='Reds', alpha=0.4)
        rect2 = plt.Rectangle((bbox[0], bbox[1]), bbox[2], bbox[3],
                              linewidth=1, edgecolor='lime', facecolor='none', linestyle='--')
        axes[row_idx, 2].add_patch(rect2)
        axes[row_idx, 2].set_title(f'MedSAM Maske\nDice={row["dice"]:.3f} IoU={row["iou"]:.3f}', fontsize=9)
        axes[row_idx, 2].axis('off')
    
    plt.suptitle('MedSAM Nodul/Kitle Segmentasyon Ornekleri', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'cxr_medsam_examples.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: cxr_medsam_examples.png")

---
## 8. CXR Siniflandirma Sonuclariyla Birlesik Analiz

In [ ]:
# DenseNet-121 (notebook 02) tahminlerini yukle
densenet_pred_path = OUTPUT_DIR / 'cxr_predictions_full.csv'
# Kaggle kernel output'larindan da arayabiliriz
kernel_output_candidates = list(INPUT_DIR.rglob('cxr_predictions_full.csv'))

densenet_df = None
if densenet_pred_path.exists():
    densenet_df = pd.read_csv(densenet_pred_path)
    print(f"DenseNet-121 tahminleri yuklendi: {len(densenet_df):,} goruntu")
elif kernel_output_candidates:
    densenet_df = pd.read_csv(kernel_output_candidates[0])
    print(f"DenseNet-121 tahminleri yuklendi (kernel): {len(densenet_df):,} goruntu")
else:
    print("DenseNet-121 tahmin dosyasi bulunamadi, birlesik analiz atlanacak.")
    print("Not: 02_cxr_pipeline_torchxrayvision notebook'unu once calistirin.")

In [ ]:
if densenet_df is not None and len(seg_df) > 0:
    # Segmentasyon sonuclarini DenseNet tahminleriyle birlestir
    # Her seg_df satirindaki goruntunun DenseNet skorunu bul
    
    # DenseNet'te Nodule prediction sutununu bul
    nodule_col = [c for c in densenet_df.columns if 'nodule' in c.lower() and 'pred' in c.lower()]
    mass_col = [c for c in densenet_df.columns if 'mass' in c.lower() and 'pred' in c.lower()]
    
    if nodule_col or mass_col:
        pred_col = nodule_col[0] if nodule_col else mass_col[0]
        print(f"DenseNet tahmin sutunu: {pred_col}")
        
        # Birlestir
        merged = seg_df.merge(
            densenet_df[['image_name', pred_col]].rename(columns={pred_col: 'densenet_score'}),
            on='image_name',
            how='left'
        )
        merged_valid = merged.dropna(subset=['densenet_score'])
        
        if len(merged_valid) > 0:
            print(f"Birlesik veri: {len(merged_valid):,} goruntu")
            
            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            
            # 1. DenseNet skor vs Segmentasyon alani
            axes[0].scatter(merged_valid['densenet_score'], merged_valid['pred_area'],
                           alpha=0.5, s=20, c='#3498db')
            axes[0].set_xlabel('DenseNet-121 Skor')
            axes[0].set_ylabel('Segmentasyon Alani (piksel)')
            axes[0].set_title('Siniflandirma Skoru vs Segmentasyon Alani', fontweight='bold')
            
            # 2. DenseNet skor vs Dice
            axes[1].scatter(merged_valid['densenet_score'], merged_valid['dice'],
                           alpha=0.5, s=20, c='#2ecc71')
            axes[1].set_xlabel('DenseNet-121 Skor')
            axes[1].set_ylabel('Dice Koefisyeni')
            axes[1].set_title('Siniflandirma Skoru vs Segmentasyon Kalitesi', fontweight='bold')
            
            # 3. Quadrant analizi: siniflandirma belirsiz ama segmentasyon net
            dn_med = merged_valid['densenet_score'].median()
            dice_med = merged_valid['dice'].median()
            
            q_colors = []
            for _, r in merged_valid.iterrows():
                if r['densenet_score'] >= dn_med and r['dice'] >= dice_med:
                    q_colors.append('#27ae60')  # Ikisi de iyi
                elif r['densenet_score'] < dn_med and r['dice'] >= dice_med:
                    q_colors.append('#e67e22')  # Siniflandirma dusuk, segmentasyon iyi
                elif r['densenet_score'] >= dn_med and r['dice'] < dice_med:
                    q_colors.append('#3498db')  # Siniflandirma iyi, segmentasyon dusuk
                else:
                    q_colors.append('#e74c3c')  # Ikisi de dusuk
            
            axes[2].scatter(merged_valid['densenet_score'], merged_valid['dice'],
                           c=q_colors, alpha=0.6, s=20)
            axes[2].axvline(dn_med, color='gray', linestyle='--', alpha=0.5)
            axes[2].axhline(dice_med, color='gray', linestyle='--', alpha=0.5)
            axes[2].set_xlabel('DenseNet-121 Skor')
            axes[2].set_ylabel('Dice')
            axes[2].set_title('Quadrant Analizi\n(Turuncu=Segmentasyon deger katiyor)', fontweight='bold', fontsize=10)
            
            # Quadrant istatistikleri
            n_seg_adds_value = sum(1 for c in q_colors if c == '#e67e22')
            print(f"\nQuadrant Analizi:")
            print(f"  Siniflandirma iyi + Segmentasyon iyi: {sum(1 for c in q_colors if c == '#27ae60')}")
            print(f"  Siniflandirma dusuk + Segmentasyon iyi: {n_seg_adds_value} (segmentasyon deger katiyor)")
            print(f"  Siniflandirma iyi + Segmentasyon dusuk: {sum(1 for c in q_colors if c == '#3498db')}")
            print(f"  Ikisi de dusuk: {sum(1 for c in q_colors if c == '#e74c3c')}")
            
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / 'cxr_medsam_classification_integration.png', dpi=150, bbox_inches='tight')
            plt.show()
            print("Saved: cxr_medsam_classification_integration.png")
        else:
            print("Birlesik veri bos -- goruntu isimleri eslesmedi.")
    else:
        print("DenseNet tahmin sutunlari bulunamadi.")
else:
    print("Birlesik analiz icin veriler yeterli degil.")

---
## 9. Akciger Bolgesi Segmentasyonu (500 Goruntu)

In [ ]:
# 500 rastgele goruntu uzerinde akciger bolgesi segmentasyonu
N_LUNG_SAMPLES = 500

all_image_names_list = list(image_index.keys())
np.random.seed(42)
lung_sample_names = np.random.choice(all_image_names_list, 
                                     size=min(N_LUNG_SAMPLES, len(all_image_names_list)), 
                                     replace=False)

print(f"Akciger segmentasyonu basliyor: {len(lung_sample_names)} goruntu")
print("-" * 60)

lung_results = []
start_time = time.time()

for i, img_name in enumerate(lung_sample_names):
    img_path = image_index.get(img_name)
    if img_path is None or not img_path.exists():
        continue
    
    try:
        t0 = time.time()
        mask = segment_lung_region(img_path, model, processor, device)
        seg_time = time.time() - t0
        
        # Goruntu boyutunu al
        img = Image.open(img_path)
        img_w, img_h = img.size
        total_pixels = img_w * img_h
        
        lung_area = mask.sum()
        lung_ratio = lung_area / total_pixels
        
        # Goruntunun etiketlerini al
        img_labels = df[df['Image Index'] == img_name]['Finding Labels'].values
        finding = img_labels[0] if len(img_labels) > 0 else 'Unknown'
        
        lung_results.append({
            'image_name': img_name,
            'finding_labels': finding,
            'img_width': img_w,
            'img_height': img_h,
            'lung_area_pixels': int(lung_area),
            'total_pixels': total_pixels,
            'lung_ratio': lung_ratio,
            'seg_time': seg_time
        })
        
    except Exception as e:
        if i < 3:
            print(f"  HATA [{img_name}]: {e}")
        continue
    
    if (i + 1) % 100 == 0 or (i + 1) == len(lung_sample_names):
        elapsed = time.time() - start_time
        speed = (i + 1) / elapsed
        print(f"  {i+1:>5}/{len(lung_sample_names)} ({(i+1)/len(lung_sample_names)*100:>5.1f}%) "
              f"| {speed:.1f} img/s")

lung_df = pd.DataFrame(lung_results)
total_time = time.time() - start_time

print(f"\n{'=' * 60}")
print(f"Akciger segmentasyonu tamamlandi!")
print(f"  Islenen: {len(lung_df):,} goruntu")
print(f"  Sure: {total_time:.1f}s ({len(lung_df)/total_time:.1f} img/s)")

In [ ]:
# Akciger alani istatistikleri
if len(lung_df) > 0:
    print("=" * 65)
    print("Akciger Bolgesi Segmentasyon Istatistikleri")
    print("=" * 65)
    
    print(f"\n{'Metrik':>25s} | {'Deger':>12s}")
    print("-" * 45)
    print(f"  {'Ortalama Lung Ratio':>25s} | {lung_df['lung_ratio'].mean():>12.4f}")
    print(f"  {'Medyan Lung Ratio':>25s} | {lung_df['lung_ratio'].median():>12.4f}")
    print(f"  {'Std Lung Ratio':>25s} | {lung_df['lung_ratio'].std():>12.4f}")
    print(f"  {'Min Lung Ratio':>25s} | {lung_df['lung_ratio'].min():>12.4f}")
    print(f"  {'Max Lung Ratio':>25s} | {lung_df['lung_ratio'].max():>12.4f}")
    print(f"  {'Ortalama Lung Area (px)':>25s} | {lung_df['lung_area_pixels'].mean():>12,.0f}")
    print(f"  {'Ortalama Seg Sure (s)':>25s} | {lung_df['seg_time'].mean():>12.3f}")
    
    # Patolojiye gore akciger alani
    lung_df['has_pathology'] = lung_df['finding_labels'].apply(lambda x: 0 if x == 'No Finding' else 1)
    
    print(f"\nPatoloji durumuna gore akciger alani:")
    for group, label in [(0, 'Normal'), (1, 'Patolojik')]:
        subset = lung_df[lung_df['has_pathology'] == group]
        if len(subset) > 0:
            print(f"  {label} (n={len(subset):,}): "
                  f"lung_ratio = {subset['lung_ratio'].mean():.4f} +/- {subset['lung_ratio'].std():.4f}")

In [ ]:
# Akciger segmentasyonu gorsellestirmesi
if len(lung_df) > 0:
    # 6 ornek goster
    n_show = min(6, len(lung_df))
    sample_indices = np.linspace(0, len(lung_df) - 1, n_show, dtype=int)
    sample_lung = lung_df.iloc[sample_indices]
    
    fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
    if n_show == 1:
        axes = axes[:, np.newaxis]
    
    for col_idx, (_, row) in enumerate(sample_lung.iterrows()):
        img_path = image_index.get(row['image_name'])
        if img_path is None or not img_path.exists():
            continue
        
        img = Image.open(img_path).convert('L')
        img_arr = np.array(img)
        
        # Segmentasyonu tekrar calistir
        try:
            mask = segment_lung_region(img_path, model, processor, device)
        except Exception:
            continue
        
        # Ust satir: orijinal
        axes[0, col_idx].imshow(img_arr, cmap='gray')
        axes[0, col_idx].set_title(f"{row['image_name'][:15]}...\n{row['finding_labels'][:20]}", fontsize=7)
        axes[0, col_idx].axis('off')
        
        # Alt satir: overlay
        axes[1, col_idx].imshow(img_arr, cmap='gray')
        axes[1, col_idx].imshow(mask, cmap='Blues', alpha=0.35)
        axes[1, col_idx].set_title(f"Lung ratio: {row['lung_ratio']:.3f}", fontsize=8)
        axes[1, col_idx].axis('off')
    
    axes[0, 0].set_ylabel('Orijinal', fontsize=11, fontweight='bold')
    axes[1, 0].set_ylabel('MedSAM Lung', fontsize=11, fontweight='bold')
    
    plt.suptitle('MedSAM Akciger Bolgesi Segmentasyonu Ornekleri', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'cxr_medsam_lung_regions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: cxr_medsam_lung_regions.png")

---
## 10. Sonuclari Kaydet

In [ ]:
# 1. Segmentasyon sonuclari CSV
if len(seg_df) > 0:
    seg_df.to_csv(OUTPUT_DIR / 'cxr_medsam_segmentation_results.csv', index=False)
    print(f"Saved: cxr_medsam_segmentation_results.csv ({len(seg_df):,} rows)")

# 2. Akciger segmentasyon istatistikleri CSV
if len(lung_df) > 0:
    lung_df.to_csv(OUTPUT_DIR / 'cxr_medsam_lung_stats.csv', index=False)
    print(f"Saved: cxr_medsam_lung_stats.csv ({len(lung_df):,} rows)")

# 3. Ozet istatistikler
summary = {
    'model': 'MedSAM ViT-Base (flaviagiammarino/medsam-vit-base)',
    'n_bbox_processed': len(seg_df) if len(seg_df) > 0 else 0,
    'n_lung_processed': len(lung_df) if len(lung_df) > 0 else 0,
    'mean_dice': seg_df['dice'].mean() if len(seg_df) > 0 else None,
    'std_dice': seg_df['dice'].std() if len(seg_df) > 0 else None,
    'mean_iou': seg_df['iou'].mean() if len(seg_df) > 0 else None,
    'std_iou': seg_df['iou'].std() if len(seg_df) > 0 else None,
    'mean_confidence': seg_df['confidence'].mean() if len(seg_df) > 0 else None,
    'mean_seg_time': seg_df['seg_time'].mean() if len(seg_df) > 0 else None,
    'mean_lung_ratio': lung_df['lung_ratio'].mean() if len(lung_df) > 0 else None,
    'speed_img_per_sec': len(seg_df) / seg_df['seg_time'].sum() if len(seg_df) > 0 and seg_df['seg_time'].sum() > 0 else None,
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(OUTPUT_DIR / 'cxr_medsam_summary.csv', index=False)
print(f"Saved: cxr_medsam_summary.csv")

# Cikti dosyalarini listele
print(f"\nTum cikti dosyalari:")
for f in sorted(OUTPUT_DIR.glob('cxr_medsam*')):
    size_kb = f.stat().st_size / 1024
    if size_kb > 1024:
        print(f"  {f.name} ({size_kb/1024:.1f} MB)")
    else:
        print(f"  {f.name} ({size_kb:.1f} KB)")

---
## 11. Final Ozet

In [ ]:
print("\n" + "=" * 70)
print("AlpCAN CXR Pipeline -- MedSAM Segmentasyon Baseline OZET")
print("=" * 70)

n_params = sum(p.numel() for p in model.parameters())
print(f"\n  Model: MedSAM ViT-Base (flaviagiammarino/medsam-vit-base)")
print(f"  Parametre: {n_params:,} ({n_params/1e6:.1f}M)")
print(f"  Device: {device}")

if len(seg_df) > 0:
    speed = len(seg_df) / seg_df['seg_time'].sum() if seg_df['seg_time'].sum() > 0 else 0
    print(f"\n  BBox-guided segmentasyon:")
    print(f"    Islenen: {len(seg_df):,} BBox annotasyonlu goruntu")
    print(f"    Ortalama Dice: {seg_df['dice'].mean():.4f} +/- {seg_df['dice'].std():.4f}")
    print(f"    Ortalama IoU:  {seg_df['iou'].mean():.4f} +/- {seg_df['iou'].std():.4f}")
    print(f"    Ortalama Confidence: {seg_df['confidence'].mean():.4f}")
    print(f"    Hiz: {speed:.1f} img/s")
    
    for finding in seg_df['finding'].unique():
        subset = seg_df[seg_df['finding'] == finding]
        print(f"    {finding} (n={len(subset)}): Dice={subset['dice'].mean():.4f}, IoU={subset['iou'].mean():.4f}")

if len(lung_df) > 0:
    print(f"\n  Akciger bolgesi segmentasyonu:")
    print(f"    Islenen: {len(lung_df):,} goruntu")
    print(f"    Ortalama akciger orani: {lung_df['lung_ratio'].mean():.4f}")

print(f"\n  Cikti dosyalari:")
for f in sorted(OUTPUT_DIR.glob('cxr_medsam*')):
    size_kb = f.stat().st_size / 1024
    print(f"    {f.name} ({size_kb:.1f} KB)")

print(f"\n  Sonraki adimlar:")
print(f"    1. MedSAM fine-tuning NIH BBox verileri uzerinde")
print(f"    2. Siniflandirma + Segmentasyon pipeline entegrasyonu")
print(f"    3. 3D LIDC-IDRI uzerinde volumetrik segmentasyon")
print("=" * 70)